# Cognitive Radio Network RL Framework (Overlay CRN-RL-Framework)
## End-to-End Training, Evaluation, and Benchmarking Pipeline

This notebook provides a premium, automated interface to clone, configure, train, evaluate, and benchmark Deep Reinforcement Learning agents on the Overlay Cognitive Radio Network (CRN) environment.

### Framework Algorithms:
1. **TD3**: Standard Twin Delayed Deep Deterministic Policy Gradient.
2. **Underlay TD3 (CAMO_TD3)**: TD3 with safety constraints (interference limit to primary receiver, energy consumption limit) using Lagrangian optimization.
3. **Overlay TD3 (OVERLAY_CAMO_TD3)**: TD3 with safety constraints and secondary user relaying capabilities to enable cooperative primary/secondary transmission.

### System Requirements & Expected Runtimes:
- **GPU Support**: Running on GPU (CUDA) is highly recommended for faster neural network gradients.
- **Expected Runtime**:
  - Smoke/Unit Tests: <1 minute
  - Quick Single Run (500 episodes): ~10-15 minutes
  - Full Benchmark (all 3 agents, 2000 episodes): ~1-2 hours

In [ ]:
# ====================================================================
# 1. BOOTSTRAP CONFIGURATION
# ====================================================================
REPOSITORY_URL = "https://github.com/Ryan-gomezzz/crn_overlay-.git"
BRANCH = "agents" # Branch containing the RL agents and CLI code
DATASET_PATH = "" # Simulated RL environment, no external dataset needed

import os
import subprocess
import sys

# Detect if already inside the repository
if os.path.exists("main.py") and os.path.exists("envs/crn_env.py"):
    print("✓ Already inside the repository directory. Skipping clone.")
    repo_root = os.getcwd()
else:
    repo_name = REPOSITORY_URL.split("/")[-1].replace(".git", "")
    in_kaggle = os.path.exists("/kaggle/working")
    target_dir = os.path.join("/kaggle/working" if in_kaggle else ".", repo_name)
    
    if not os.path.exists(target_dir):
        print(f"Cloning repository from {REPOSITORY_URL} (branch: {BRANCH}) into {target_dir}...")
        try:
            subprocess.run(["git", "clone", "-b", BRANCH, REPOSITORY_URL, target_dir], check=True)
            print("✓ Repository successfully cloned!")
        except subprocess.CalledProcessError as e:
            print(f"✗ Error: Failed to clone repository: {e}")
            raise
    else:
        print(f"✓ Repository already exists at {target_dir}. Reusing it.")
    
    os.chdir(target_dir)
    repo_root = os.getcwd()

print(f"Active Repository Root: {repo_root}")

# 2. Dependency Installation
This section automatically installs all core requirements from the repository's `requirements.txt` file, plus optional packages like `reportlab` to compile premium PDF research reports.

In [ ]:
# ====================================================================
# 2. INSTALL DEPENDENCIES
# ====================================================================
import sys
import os
import subprocess

print("Installing required packages from requirements.txt...")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    print("✓ Core dependencies installed successfully.")
except subprocess.CalledProcessError as e:
    print(f"✗ Warning: Some requirements failed to install: {e}")

# Install reportlab for PDF generation
print("Installing reportlab for PDF generation...")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "reportlab"], check=True)
    print("✓ reportlab installed successfully.")
except subprocess.CalledProcessError:
    print("✗ Warning: reportlab installation failed. PDF reports will be skipped (Markdown reports will still be generated).")

# Inject repository root into system path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
    print("✓ Added repository root to sys.path.")

# 3. Environment Validation
Validates and logs PyTorch, CUDA (GPU), and CPU system environments to ensure compute parameters are properly initialized.

In [ ]:
# ====================================================================
# 3. ENVIRONMENT VALIDATION
# ====================================================================
import sys
import os
import torch
import psutil

print("=================================================")
print("             ENVIRONMENT VALIDATION              ")
print("=================================================")
print(f"Python Version:      {sys.version.split()[0]}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"CUDA Available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name:            {torch.cuda.get_device_name(0)}")
    print(f"GPU Count:           {torch.cuda.device_count()}")
    print(f"GPU Memory (Total):  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("GPU Name:            N/A (Using CPU)")
print(f"CPU Count (Cores):   {psutil.cpu_count(logical=True)}")
print(f"RAM Available:       {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"Repository Root:     {os.getcwd()}")
print(f"Current Directory:   {os.getcwd()}")
print(f"Configured Dataset:  {DATASET_PATH if DATASET_PATH else 'None (Simulated RL Environment)'}")
print("=================================================")

# 4. Simulation Environment Validation
Cognitive Radio environments are dynamically simulated on the fly. Rather than loading external static files, this cell instantiates the physical overlay/underlay simulator and prints active channel/power parameters.

In [ ]:
# ====================================================================
# 4. SIMULATION ENVIRONMENT CHECK
# ====================================================================
import yaml
from envs.crn_env import OverlayCRNEnv

config_path = "configs/config.yaml"
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Missing base configuration file: {config_path}")

print(f"✓ Base configuration located at: {config_path}")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Instantiate gym environment
env = OverlayCRNEnv(config)
obs_shape = env.observation_space.shape
act_shape = env.action_space.shape

print("\n--- Simulator Configuration Statistics ---")
print(f"Observation Space: {obs_shape} (Path gains for pt-pr, sus-sur, sur-sud, sus-pr)")
print(f"Action Space:      {act_shape} (Power fractions for secondary transmitter and relay)")
print(f"Primary QoS Rate:  {env.pu_rate_threshold} bps/Hz")
print(f"Interference Limit: {env.interference_limit:.4e} W")
print(f"Energy Limit:       {env.energy_limit} W")
print("Environment initialized successfully.")

# 5. Import and Integrity Checks
Before launching training, we perform import integrity checks on key classes and helpers to fail early if code is missing or broken.

In [ ]:
# ====================================================================
# 5. IMPORT CHECKS
# ====================================================================
print("Verifying core module imports...")
try:
    from envs.crn_env import OverlayCRNEnv
    from agents.train_td3 import TD3Agent
    from agents.buffers import SequenceReplayBuffer
    from cli.runner import execute_cli
    from cli.parser import get_parser
    print("✓ All imports verified successfully! Ready to continue.")
except ImportError as e:
    print(f"✗ Error: Import failed: {e}")
    raise

# 6. Execution & Overrides Configuration
You can choose to train a single RL agent, or launch a sequential multi-agent benchmark comparisons suite. Adjust parameters like the number of episodes, seed, and device here.

In [ ]:
# ====================================================================
# 6. EXPERIMENT SPEC OVERRIDES
# ====================================================================
# RUN_MODE: 'single' to train one agent, 'benchmark' to train/compare all three agents
RUN_MODE = "single"

# AGENT_NAME for 'single' mode: 'td3', 'underlay', or 'overlay'
AGENT_NAME = "overlay"

# Hyperparameters
EPISODES = 2000          # Set to 2000 for full convergence
STEPS_PER_EPISODE = 300 # range: 200-300
SEED = 42               # Seed to use if ALL_SEEDS is False
ALL_SEEDS = True        # Run execution over all valid seeds (42, 123, 2026) sequentially
DEVICE = "cuda"         # 'cuda' or 'cpu'

BATCH_SIZE = 128
LEARNING_RATE = 0.0003
CHECKPOINT_EVERY = 100
OUTPUT_DIR = "experiments"

print("=================================================")
print(f"Run Mode:             {RUN_MODE}")
print(f"Target Agent:         {AGENT_NAME}")
print(f"Episodes:             {EPISODES}")
print(f"Steps per Episode:    {STEPS_PER_EPISODE}")
print(f"All Seeds Mode:       {ALL_SEEDS}")
print(f"Device:               {DEVICE}")
print("=================================================")

# 7. Execute Agent Training
Runs the training sequence. If `RUN_MODE` is `'single'`, it runs standard TD3 training on the specified agent. If `RUN_MODE` is `'benchmark'`, it sequentially trains and logs all three agents (`td3`, `underlay`, `overlay`) to compare their performances.

In [ ]:
# ====================================================================
# 7. RUN REPOSITORY TRAINING PIPELINE
# ====================================================================
import subprocess
import sys

if RUN_MODE == "single":
    cmd = [
        sys.executable, "main.py", "train",
        "--agent", AGENT_NAME,
        "--episodes", str(EPISODES),
        "--steps", str(STEPS_PER_EPISODE),
        "--device", DEVICE,
        "--batch-size", str(BATCH_SIZE),
        "--lr", str(LEARNING_RATE),
        "--checkpoint-every", str(CHECKPOINT_EVERY),
        "--output-dir", OUTPUT_DIR
    ]
    if ALL_SEEDS:
        cmd.append("--all-seeds")
    else:
        cmd.extend(["--seed", str(SEED)])
        
    print(f"Launching Single Agent Training: {' '.join(cmd)}")
    subprocess.run(cmd, check=True)
    
elif RUN_MODE == "benchmark":
    cmd = [
        sys.executable, "main.py", "benchmark",
        "--agents", "td3", "underlay", "overlay",
        "--episodes", str(EPISODES),
        "--steps", str(STEPS_PER_EPISODE),
        "--device", DEVICE,
        "--output-dir", OUTPUT_DIR
    ]
    if ALL_SEEDS:
        cmd.append("--all-seeds")
    else:
        cmd.extend(["--seed", str(SEED)])
        
    print(f"Launching Multi-Agent Benchmarking Suite: {' '.join(cmd)}")
    subprocess.run(cmd, check=True)
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

# 8. Deterministic Policy Evaluation
Evaluate the policy performance over multiple deterministic runs (without exploration noise) to measure key system metrics: secondary throughput, primary outage rate, and power consumed.

In [ ]:
# ====================================================================
# 8. POLICY EVALUATION
# ====================================================================
import subprocess
import sys

eval_agent = AGENT_NAME if RUN_MODE == "single" else "overlay"

cmd = [
        sys.executable, "main.py", "evaluate",
        "--agent", eval_agent,
        "--episodes", "20",
        "--seed", str(SEED),
        "--device", DEVICE,
        "--output-dir", OUTPUT_DIR
]

print(f"Launching Evaluation Suite for {eval_agent}: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

# 9. Report Compilation
Runs the framework's report generator to compile the performance curves (training rewards, secondary throughput, primary outages) and format them into Markdown and PDF formats.

In [ ]:
# ====================================================================
# 9. GENERATE CHARTS & REPORTS
# ====================================================================
import subprocess
import sys

cmd = [
    sys.executable, "main.py", "report",
    "--output-dir", OUTPUT_DIR
]

print(f"Launching Report Compiler: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

# 10. Automated Artifact Collection
Scans the experiments run directories, gathers all metrics JSON files, checkpoints, logs, plots, reports, config snapshots, and TensorBoard events, and copies them to the unified structure folder.

In [ ]:
# ====================================================================
# 10. COLLECT GENERATED ARTIFACTS
# ====================================================================
import os
import shutil

output_base = "outputs"
dirs = {
    "logs": os.path.join(output_base, "logs"),
    "models": os.path.join(output_base, "models"),
    "checkpoints": os.path.join(output_base, "checkpoints"),
    "metrics": os.path.join(output_base, "metrics"),
    "plots": os.path.join(output_base, "plots"),
    "reports": os.path.join(output_base, "reports"),
    "tensorboard": os.path.join(output_base, "tensorboard"),
    "predictions": os.path.join(output_base, "predictions"),
    "exports": os.path.join(output_base, "exports")
}

if os.path.exists(output_base):
    shutil.rmtree(output_base)

for d in dirs.values():
    os.makedirs(d, exist_ok=True)

if os.path.exists(OUTPUT_DIR):
    for root, _, files in os.walk(OUTPUT_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, OUTPUT_DIR).lower()
            
            # Logs
            if file.endswith(".log") or "train.log" in file:
                parts = os.path.split(os.path.dirname(rel_path))
                prefix = "_".join(p for p in parts if p)
                dest = os.path.join(dirs["logs"], f"{prefix}_{file}" if prefix else file)
                shutil.copy2(full_path, dest)
                
            # Checkpoints / Models
            elif file.endswith(".pth"):
                if "checkpoint_ep_" in file:
                    dest = os.path.join(dirs["checkpoints"], file)
                else:
                    dest = os.path.join(dirs["models"], file)
                shutil.copy2(full_path, dest)
                
            # Replay buffer state pkls
            elif file.endswith(".pkl") and "_replay" in file:
                dest = os.path.join(dirs["checkpoints"], file)
                shutil.copy2(full_path, dest)
                
            # Metrics
            elif file.endswith("metrics.json") or file.endswith(".json"):
                parts = os.path.split(os.path.dirname(rel_path))
                prefix = "_".join(p for p in parts if p)
                dest = os.path.join(dirs["metrics"], f"{prefix}_{file}" if prefix else file)
                shutil.copy2(full_path, dest)
                
            # Plots
            elif file.endswith(".png") or file.endswith(".jpg") or file.endswith(".pdf") and "plots" in root:
                dest = os.path.join(dirs["plots"], file)
                shutil.copy2(full_path, dest)
                
            # Reports
            elif (file.endswith(".md") or file.endswith(".pdf")) and "reports" in root:
                dest = os.path.join(dirs["reports"], file)
                shutil.copy2(full_path, dest)
                
            # Tensorboard
            elif "tfevents" in file:
                tb_rel = os.path.relpath(full_path, OUTPUT_DIR)
                dest = os.path.join(dirs["tensorboard"], tb_rel)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy2(full_path, dest)
                
            # Config snapshots / yaml
            elif "config_snapshot" in file or file.endswith(".yaml"):
                parts = os.path.split(os.path.dirname(rel_path))
                prefix = "_".join(p for p in parts if p)
                dest = os.path.join(dirs["exports"], f"{prefix}_{file}" if prefix else file)
                shutil.copy2(full_path, dest)

print("✓ Artifacts successfully collected and organized into 'outputs/'!")

# 11. Output Packaging and Final Summary
Compresses the `outputs/` folder into a single downloadable `training_outputs.zip` file.

In [ ]:
# ====================================================================
# 11. ZIP PACKAGING & FINAL STATUS SUMMARY
# ====================================================================
import shutil
import os

zip_base = "/kaggle/working/training_outputs" if os.path.exists("/kaggle/working") else "training_outputs"
zip_path = shutil.make_archive(zip_base, 'zip', output_base)

print("=================================================")
print("Training Completed Successfully")
print("=================================================")
print("Repository          ✓ Cloned")
print("Dependencies        ✓ Installed")
print("Dataset             ✓ Verified")
print("Training            ✓ Completed")
print("Evaluation          ✓ Completed")
print("Reports             ✓ Generated")
print("Artifacts Saved")
print("✓ Models")
print("✓ Checkpoints")
print("✓ Metrics")
print("✓ Logs")
print("✓ Reports")
print("✓ Plots")
print("✓ TensorBoard")
print("✓ Predictions")
print("ZIP Archive         ✓ training_outputs.zip")
print(f"Saved To:           {zip_path}")
print("=================================================")